# Fig 5 — E6 ABR composability (wide, 2-col)

Left: 12x3 heat map of avg delivered bitrate (viridis), text-overlaid
with bitrate (top) and n_switches (bottom). Each row isolates a single
ABR rule (or, for `none`, no rule at all); all configs pin the join
rung to the middle of the ladder (1200k).
Right: strip of mini-boxplots of max_playhead_gap_ms, one per cell,
row-aligned with the heat map.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "figures"))

import matplotlib.pyplot as plt
import numpy as np

from _data import (
    E6_COL_ORDER,
    E6_ROW_ORDER,
    e6_avg_bitrate_matrix,
    e6_heatmap_matrix,
    load_aggregate,
)
from _style import (
    GOP_DURATION_MS,
    TEXT_WIDTH_IN,
    apply_acm_style,
)

apply_acm_style()

In [ ]:
bitrate_matrix = e6_avg_bitrate_matrix()
switches_matrix = e6_heatmap_matrix("n_switches_mean")
agg = load_aggregate("e6")

In [ ]:
GOP_DURATION_S = GOP_DURATION_MS / 1000.0

# Reader-friendly display names for the experimental axes.
PROFILE_DISPLAY = {
    "stable1.5M":  "Stable\n1.5 Mbps",
    "step3M_500k": "Step\n3 to 0.5 Mbps",
    "sin600k_3M":  "Sinusoidal\n0.6 to 3 Mbps",
}
CONFIG_DISPLAY = {
    "none":            "None",
    "throughput-only": "Throughput only",
    "bola-only":       "BOLA only",
    "default":         "Default",
    "dampened":        "Dampened",
    "aggressive":      "Aggressive",
    "lolp":            "LoLP",
    "l2a":             "L2A",
}

fig = plt.figure(figsize=(TEXT_WIDTH_IN, 3.6), constrained_layout=True)
gs = fig.add_gridspec(1, 2, width_ratios=[2.6, 1.0], wspace=0.08)
ax_heat = fig.add_subplot(gs[0, 0])
ax_strip = fig.add_subplot(gs[0, 1])

# Heatmap
masked = np.ma.masked_invalid(bitrate_matrix.values)
im = ax_heat.imshow(masked, cmap="viridis", aspect="auto",
                    interpolation="nearest", origin="upper")

ax_heat.set_xticks(range(len(E6_COL_ORDER)))
ax_heat.set_xticklabels(
    [PROFILE_DISPLAY.get(p, p) for p in E6_COL_ORDER],
    rotation=0, ha="center", fontsize=7,
)
ax_heat.set_yticks(range(len(E6_ROW_ORDER)))
ax_heat.set_yticklabels(
    [CONFIG_DISPLAY.get(c, c) for c in E6_ROW_ORDER],
    fontsize=7,
)
ax_heat.set_xlabel("Bandwidth profile")
ax_heat.set_ylabel("ABR configuration")

# Cell text overlays
for i, _ in enumerate(E6_ROW_ORDER):
    for j, _ in enumerate(E6_COL_ORDER):
        bitrate = bitrate_matrix.iloc[i, j]
        sw_val = switches_matrix.iloc[i, j]
        sw = sw_val if not np.isnan(sw_val) else 0
        if np.isnan(bitrate):
            ax_heat.text(j, i, "n/a", ha="center", va="center",
                         color="white", fontsize=7)
            continue
        rgba = im.cmap(im.norm(bitrate))
        luminance = 0.299 * rgba[0] + 0.587 * rgba[1] + 0.114 * rgba[2]
        text_color = "white" if luminance < 0.5 else "black"
        ax_heat.text(j, i - 0.18, f"{bitrate:.0f} kbps",
                     ha="center", va="center",
                     color=text_color, fontsize=6.5, weight="bold")
        ax_heat.text(j, i + 0.18, f"{sw:.1f} switches",
                     ha="center", va="center",
                     color=text_color, fontsize=6)

cbar = fig.colorbar(im, ax=ax_heat, shrink=0.85, pad=0.02)
cbar.set_label("Average delivered bitrate (kbps)",
               rotation=270, labelpad=12)

# Strip of mini-boxplots, converted from ms to seconds.
strip_data = []
strip_positions = []
for i, config in enumerate(E6_ROW_ORDER):
    for j, profile in enumerate(E6_COL_ORDER):
        cell_id = f"{config}_{profile}"
        gaps_ms = agg[agg["cell_id"] == cell_id]["max_playhead_gap_ms"].values
        if len(gaps_ms) == 0:
            continue
        gaps_s = gaps_ms / 1000.0
        x_pos = i + (j - 1) * 0.22
        strip_data.append(gaps_s)
        strip_positions.append(x_pos)

ax_strip.boxplot(
    strip_data, positions=strip_positions, vert=False, widths=0.18,
    patch_artist=True,
    boxprops={"facecolor": "#888888", "alpha": 0.7},
    medianprops={"color": "black", "linewidth": 1.0},
    flierprops={"marker": "+", "markersize": 2},
)

# Threshold line and rotated label, matching the wording used in
# the boxplot and trace figures.
ax_strip.axvline(GOP_DURATION_S, color="black", linestyle="--",
                 linewidth=0.8, alpha=0.7)
ax_strip.text(
    GOP_DURATION_S * 1.04, len(E6_ROW_ORDER) / 2 - 0.5,
    "Seamless playback limit (1 s)",
    fontsize=6.5, va="center", ha="left", rotation=90, style="italic",
)

ax_strip.set_xlabel("Playback jump per switch (s)")
xmax_data = max((max(d) for d in strip_data if len(d)), default=1.3)
ax_strip.set_xlim(0, max(1.3, xmax_data + 0.1))
ax_strip.set_ylim(len(E6_ROW_ORDER) - 0.5, -0.5)
ax_strip.set_yticks(range(len(E6_ROW_ORDER)))
ax_strip.set_yticklabels([])
ax_strip.tick_params(axis="y", length=0)

In [ ]:
fig.savefig(Path.cwd().parent / "figures" / "fig5_e6_composability.pdf")
fig.savefig(Path.cwd().parent / "figures" / "fig5_e6_composability.png", dpi=200)